In [2]:
# Libraries import karo
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = 'notebook_connected'

# Data load karo
df = pd.read_csv(r'C:\Users\pc\Documents\DATA SCIENCE PROJECTS\Stock Market Price Prediction & Analysis System\data\reliance_featured_data.csv',
                 index_col='Date',
                 parse_dates=True)

print("Data loaded! ✅")
print("Shape:", df.shape)

Data loaded! ✅
Shape: (1335, 30)


In [3]:
# Target variable banao
df['Target'] = df['Close'].shift(-1)
df = df.dropna()

# Features aur Target alag karo
X = df.drop(columns=['Target'])
y = df['Target']

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=False
)

# Scaling
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()

X_test_scaled = scaler_X.transform(X_test)
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1)).ravel()

print("Preprocessing complete! ✅")
print(f"Training rows: {X_train.shape[0]}")
print(f"Testing rows:  {X_test.shape[0]}")

Preprocessing complete! ✅
Training rows: 1067
Testing rows:  267


CROSS VALIDATION

Normal train/test split mein ek baar test hota hai!
Cross Validation mein 5 baar alag alag split karke test karte hain — phir average nikalte hain!
Socho aise — 5 alag exams deke average nikalo — zyada reliable result milega! 😊

In [4]:
# Cross Validation — Linear Regression
from sklearn.model_selection import KFold

print("Cross Validation Results:")
print("=" * 40)

kf = KFold(n_splits=5, shuffle=False)

# Linear Regression
lr = LinearRegression()
lr_cv_scores = cross_val_score(lr, X_train_scaled, y_train_scaled, 
                                cv=kf, scoring='r2')

print(f"\nLinear Regression:")
print(f"CV Scores: {lr_cv_scores.round(4)}")
print(f"Mean R²:   {lr_cv_scores.mean():.4f}")
print(f"Std:       {lr_cv_scores.std():.4f}")

Cross Validation Results:

Linear Regression:
CV Scores: [0.9344 0.8666 0.9311 0.9824 0.9686]
Mean R²:   0.9366
Std:       0.0401


In [5]:
# Ridge aur Lasso try karo
from sklearn.linear_model import Ridge, Lasso

# Ridge Regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train_scaled)
ridge_pred = scaler_y.inverse_transform(
    ridge.predict(X_test_scaled).reshape(-1,1)
).ravel()

ridge_r2   = r2_score(y_test, ridge_pred)
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))
ridge_mape = np.mean(np.abs((y_test - ridge_pred) / y_test)) * 100

# Lasso Regression
lasso = Lasso(alpha=0.001)
lasso.fit(X_train_scaled, y_train_scaled)
lasso_pred = scaler_y.inverse_transform(
    lasso.predict(X_test_scaled).reshape(-1,1)
).ravel()

lasso_r2   = r2_score(y_test, lasso_pred)
lasso_rmse = np.sqrt(mean_squared_error(y_test, lasso_pred))
lasso_mape = np.mean(np.abs((y_test - lasso_pred) / y_test)) * 100

print("Ridge vs Lasso Results:")
print("=" * 40)
print(f"\nRidge Regression:")
print(f"   R²:   {ridge_r2:.4f}")
print(f"   RMSE: ₹{ridge_rmse:.2f}")
print(f"   MAPE: {ridge_mape:.2f}%")

print(f"\nLasso Regression:")
print(f"   R²:   {lasso_r2:.4f}")
print(f"   RMSE: ₹{lasso_rmse:.2f}")
print(f"   MAPE: {lasso_mape:.2f}%")

Ridge vs Lasso Results:

Ridge Regression:
   R²:   0.9567
   RMSE: ₹18.87
   MAPE: 1.00%

Lasso Regression:
   R²:   0.9546
   RMSE: ₹19.31
   MAPE: 1.03%


In [6]:
# XGBoost Hyperparameter Tuning
print("XGBoost Hyperparameter Tuning...")
print("Thoda time lagega... ⏳")

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0]
}

xgb_tuned = xgb.XGBRegressor(random_state=42, n_jobs=-1)

grid_search = GridSearchCV(
    xgb_tuned,
    param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=0
)

grid_search.fit(X_train_scaled, y_train_scaled)

print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"Best CV R²: {grid_search.best_score_:.4f}")

XGBoost Hyperparameter Tuning...
Thoda time lagega... ⏳

Best Parameters: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}
Best CV R²: 0.2328


In [7]:
# Tuned XGBoost manually test karo
xgb_best = xgb.XGBRegressor(
    learning_rate=0.05,
    max_depth=3,
    n_estimators=100,
    subsample=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_best.fit(X_train_scaled, y_train_scaled)

xgb_best_pred = scaler_y.inverse_transform(
    xgb_best.predict(X_test_scaled).reshape(-1,1)
).ravel()

xgb_best_r2   = r2_score(y_test, xgb_best_pred)
xgb_best_rmse = np.sqrt(mean_squared_error(y_test, xgb_best_pred))
xgb_best_mape = np.mean(np.abs((y_test - xgb_best_pred) / y_test)) * 100

print("Tuned XGBoost Results:")
print("=" * 40)
print(f"R²:   {xgb_best_r2:.4f}")
print(f"RMSE: ₹{xgb_best_rmse:.2f}")
print(f"MAPE: {xgb_best_mape:.2f}%")

Tuned XGBoost Results:
R²:   0.9489
RMSE: ₹20.50
MAPE: 1.14%


In [8]:
# Teeno best models ka final comparison
print("FINAL MODEL COMPARISON")
print("=" * 50)

models_comparison = {
    'Linear Regression': {
        'R²': 0.9564,
        'RMSE': 18.94,
        'MAPE': 1.02
    },
    'Ridge Regression': {
        'R²': 0.9567,
        'RMSE': 18.87,
        'MAPE': 1.00
    },
    'Lasso Regression': {
        'R²': 0.9546,
        'RMSE': 19.31,
        'MAPE': 1.03
    },
    'Tuned XGBoost': {
        'R²': 0.9489,
        'RMSE': 20.50,
        'MAPE': 1.14
    },
    'Random Forest': {
        'R²': 0.9248,
        'RMSE': 24.86,
        'MAPE': 1.39
    }
}

# DataFrame banao aur sort karo
comparison_df = pd.DataFrame(models_comparison).T
comparison_df = comparison_df.sort_values('R²', ascending=False)

print(comparison_df)
print("\n" + "=" * 50)
print(f"🏆 BEST MODEL: {comparison_df.index[0]}")
print("=" * 50)

FINAL MODEL COMPARISON
                       R²   RMSE  MAPE
Ridge Regression   0.9567  18.87  1.00
Linear Regression  0.9564  18.94  1.02
Lasso Regression   0.9546  19.31  1.03
Tuned XGBoost      0.9489  20.50  1.14
Random Forest      0.9248  24.86  1.39

🏆 BEST MODEL: Ridge Regression


💡 Business Insight #13:

Ridge Regression sabse best model nikla!
Sirf 1% MAPE — matlab ₹1400 stock pe prediction sirf ₹14 galat hogi!
Ye industry standard se bahut better hai! 😎

In [9]:
# Ridge model save karo
models_path = r'C:\Users\pc\Documents\DATA SCIENCE PROJECTS\Stock Market Price Prediction & Analysis System\models'

with open(f'{models_path}\\ridge_model.pkl', 'wb') as f:
    pickle.dump(ridge, f)

# Scalers bhi save karo
with open(f'{models_path}\\scaler_X.pkl', 'wb') as f:
    pickle.dump(scaler_X, f)

with open(f'{models_path}\\scaler_y.pkl', 'wb') as f:
    pickle.dump(scaler_y, f)

print("Ridge Model saved! ✅")
print("Scalers saved! ✅")

Ridge Model saved! ✅
Scalers saved! ✅


# Day 9 - Model Improvement

## Cross Validation
- 5-Fold CV on Linear Regression
- Mean R²: 0.9366
- Std: 0.0401 — Consistent model!

## Models Tested

### Ridge Regression 🏆
- R²:   0.9567
- RMSE: ₹18.87
- MAPE: 1.00%

### Lasso Regression
- R²:   0.9546
- RMSE: ₹19.31
- MAPE: 1.03%

### Tuned XGBoost
- Best Params: learning_rate=0.05, max_depth=3
- R²:   0.9489
- RMSE: ₹20.50
- MAPE: 1.14%

## Final Decision
- Ridge Regression = BEST MODEL
- Deployment mein Ridge use karenge
- Sath mein RF aur XGBoost ka bhi option denge user ko